# 05 - Checker Tools

Validate the five checker tools against a presentation built on the fly.  
No LLM API key needed - checkers are pure python-pptx analysis.

In [ ]:
import sys
from pathlib import Path
from pprint import pprint

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "python").exists():
    if (REPO_ROOT.parent / "python").exists():
        REPO_ROOT = REPO_ROOT.parent
    else:
        raise RuntimeError("Run notebook from repo root or notebooks/ directory")

PYTHON_ROOT = REPO_ROOT / "python"
if str(PYTHON_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_ROOT))

from service import PowerPointService
from errors import BridgeError

ARTIFACT_DIR = REPO_ROOT / "artifacts" / "notebook-tests"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

service = PowerPointService()
engine_info = service.dispatch("pptx_get_engine_info", {})
print("Engine:", engine_info.get("engine"))

## Build a test presentation

Create a small deck with deliberate issues: an out-of-bounds shape, overlapping shapes,
an empty placeholder, and default template text.

In [ ]:
# Create presentation
create = service.dispatch("pptx_create_presentation", {"width": "10in", "height": "7.5in"})
pid = create["presentation_id"]
print("Presentation ID:", pid)

# Add a slide
layouts = service.dispatch("pptx_get_layouts", {"presentation_id": pid})["layouts"]
layout_name = layouts[0]["name"]
add_result = service.dispatch("pptx_add_slide", {"presentation_id": pid, "layout_name": layout_name})
slide_idx = int(add_result["added_slide_index"])
print(f"Added slide {slide_idx} with layout '{layout_name}'")

In [ ]:
# Add an out-of-bounds shape (extends past right edge of 10in slide)
service.dispatch("pptx_add_shape", {
    "presentation_id": pid,
    "slide_index": slide_idx,
    "shape_type": "rectangle",
    "left": "9.5in",
    "top": "1in",
    "width": "2in",
    "height": "1in",
    "fill_hex": "FF0000",
    "text": "Out of bounds",
})
print("Added out-of-bounds shape")

# Add two overlapping shapes
service.dispatch("pptx_add_shape", {
    "presentation_id": pid,
    "slide_index": slide_idx,
    "shape_type": "rectangle",
    "left": "2in",
    "top": "3in",
    "width": "3in",
    "height": "2in",
    "fill_hex": "00FF00",
    "text": "Shape A",
})
service.dispatch("pptx_add_shape", {
    "presentation_id": pid,
    "slide_index": slide_idx,
    "shape_type": "oval",
    "left": "3in",
    "top": "3.5in",
    "width": "3in",
    "height": "2in",
    "fill_hex": "0000FF",
    "text": "Shape B",
})
print("Added two overlapping shapes")

## 1. Position checker

In [ ]:
pos_result = service.dispatch("pptx_check_positions", {
    "presentation_id": pid,
    "check_overlaps": True,
    "check_bounds": True,
    "tolerance_px": 5,
})
print("Position check summary:")
pprint(pos_result["summary"])
print("\nIssues:")
for issue in pos_result["issues"]:
    print(f"  [{issue['severity']}] {issue['issue_type']}: {issue['description'][:100]}")

## 2. Visual consistency checker

In [ ]:
vis_result = service.dispatch("pptx_check_visual_consistency", {
    "presentation_id": pid,
    "check_fonts": True,
    "check_colors": True,
    "check_sizes": True,
})
print("Visual consistency summary:")
pprint(vis_result["summary"])
print("\nFonts used:", vis_result["font_report"]["fonts_used"])
print("Colors used:", vis_result["color_report"]["colors_used"])

## 3. Content checker

In [ ]:
content_result = service.dispatch("pptx_check_content", {
    "presentation_id": pid,
    "check_empty_placeholders": True,
    "check_default_text": True,
})
print("Content check summary:")
pprint(content_result["summary"])
if content_result["empty_placeholders"]:
    print("\nEmpty placeholders:")
    for ep in content_result["empty_placeholders"]:
        print(f"  Slide {ep['slide_index']}: {ep['placeholder_name']}")
if content_result["default_text_remaining"]:
    print("\nDefault text found:")
    for dt in content_result["default_text_remaining"]:
        print(f"  Slide {dt['slide_index']}: {dt['placeholder_name']} -> '{dt['text']}'")

## 4. Template conformance checker

Save the current deck as a "template" reference, then compare against itself (should score ~1.0).

In [ ]:
template_path = str((ARTIFACT_DIR / "nb05_ref_template.pptx").resolve())
service.dispatch("pptx_save_presentation", {
    "presentation_id": pid,
    "output_path": template_path,
})
print("Saved reference template:", template_path)

conf_result = service.dispatch("pptx_check_template_conformance", {
    "presentation_id": pid,
    "template_path": template_path,
    "check_theme": True,
    "check_layouts": True,
})
print(f"\nConformance score: {conf_result['conformance_score']}")
pprint(conf_result["summary"])
if conf_result["issues"]:
    print("\nIssues:")
    for issue in conf_result["issues"]:
        print(f"  [{issue['severity']}] {issue['issue_type']}: {issue['description']}")

## 5. Diff presentations

Open a second copy, mutate it, then diff the two.

In [ ]:
# Open the saved file as a second presentation
open_result = service.dispatch("pptx_open_presentation", {"file_path": template_path})
pid_b = open_result["presentation_id"]
print("Opened second presentation:", pid_b)

# Add a new slide to make them different
service.dispatch("pptx_add_slide", {
    "presentation_id": pid_b,
    "layout_name": layout_name,
})

# Modify text to create content difference on the existing slide
service.dispatch("pptx_add_text_box", {
    "presentation_id": pid_b,
    "slide_index": slide_idx,
    "left": "1in",
    "top": "1in",
    "width": "4in",
    "height": "0.5in",
    "text_content": "This text was added in the modified version",
})
print("Modified second presentation")

diff_result = service.dispatch("pptx_diff_presentations", {
    "presentation_id_a": pid,
    "presentation_id_b": pid_b,
    "deep_diff": True,
})
print("\nDiff summary:", diff_result["summary"])
print("Slide count:", diff_result["slide_count_diff"])
print("Added slides:", diff_result["added_slides"])
if diff_result["modified_slides"]:
    print("\nModified slides:")
    for ms in diff_result["modified_slides"]:
        print(f"  Slide {ms['slide_index']}: {len(ms['changes'])} changes")
        for ch in ms["changes"][:5]:
            print(f"    {ch}")

## Cleanup

In [ ]:
service.dispatch("pptx_close_presentation", {"presentation_id": pid})
service.dispatch("pptx_close_presentation", {"presentation_id": pid_b})
service.shutdown()
print("All sessions closed.")